# EVE 310 - Lab 06: Regression features — activity

**Module 2 | 10/01/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab06-regression-features/notebooks/lab06-activity.ipynb)

## Learning objectives

By the end of this lab you will be able to:

1. Remove outliers with the 3 standard deviation rule and say how many rows it removed, and why
2. Build an interaction term and standardize columns, and convert between kWh and standardized values
3. Read a `get_dummies` table: which category is the baseline, and what the row of a given day looks like
4. Fit a model with eight predictors and explain what the beta of a dummy column means in kWh

## Before you start

1. Click **Copy to Drive** at the top of this window and work in the copy that opens. Colab throws away anything you did not copy when the runtime ends.
2. Run the setup cell below before anything else. It creates `DATA_DIR` and `FIGURES_DIR` and downloads this lab's data files.
3. Work down the notebook in order. Replace every `--` with your code and keep the variable names exactly as given. A `--` on a line by itself means you write one or more lines of code there. This activity uses the **ECJ** building and differs from the tutorial in three places: the interaction term uses the **squared** monthly median, the columns are **standardized** (not min-max normalized), and the split is 75% / 25% with `random_state=40`. Several questions ask for numbers the tutorial never computes.
4. Questions 1 and 2 overwrite `e_df`. If you need to re-run one of them, re-run the notebook from the data cell first (**Runtime > Run before** helps).
5. When you are done: **Runtime > Restart session and run all**, then **File > Download > Download .ipynb**, and upload that file to Gradescope. The Gradescope autograder gives you feedback on each question, but **this submission is not graded** and does not count toward your final grade. The Lab 6 quiz on Canvas is what counts, and it asks about what you compute here, so finish the activity before you take it.

## 0. Setup


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# EVE 310 setup - run this cell first, every time you open this notebook.
# It downloads this lab's data files into a "data" folder in your Colab session.
import pathlib
import urllib.request

LAB = "lab06-regression-features"
DATA_FILES = ["ECJ_Energy_Lab6Activity.csv", "JES_Energy_Lab6Tutorial.csv"]

DATA_DIR = pathlib.Path("data")
FIGURES_DIR = pathlib.Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

BASE_URL = f"https://raw.githubusercontent.com/ThyanRevolter/eve310-fall-2026/main/labs/{LAB}/data"
for name in DATA_FILES:
    if not (DATA_DIR / name).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", DATA_DIR / name)

print(f"Ready. {len(DATA_FILES)} data file(s) in {DATA_DIR.resolve()}")

import datetime as dt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

**The data (run as is)**

Daily energy use of the ECJ building. `eU` is the energy use in kWh, `avgTemp` the average outdoor temperature (°F), `MonMed` the median daily energy use of that month (kWh), `classSession` is 1 when classes are in session, `dayType` is `wk` (weekday) or `wknd` (weekend), and `season` is one of four seasons.

In [ ]:
e_df = pd.read_csv(DATA_DIR / 'ECJ_Energy_Lab6Activity.csv')
e_df['allDate'] = pd.to_datetime(e_df['allDate'])
e_df.info()
e_df.head()

## Your turn

**Question 1 — Find and remove the outliers**

Plot `eU` against the date and look at it: are there values that cannot be real? Then apply the rule from the tutorial: a row is an outlier when its `eU` is more than 3 standard deviations above or below the mean `eU`. Store:

- `eu_mean`, `eu_std`: the mean and the (sample) standard deviation of `eU`, **before** anything is removed
- `upper`, `lower`: the two thresholds
- `n_high`, `n_low`: how many rows lie above `upper` and how many lie below `lower`
- `e_df`: the dataframe without the outliers, and `n_removed`: how many rows were removed

*Hint: a comparison such as `e_df['eU'] > upper` gives True/False for every row, and `.sum()` counts the True values.*

In [ ]:
# Plot eU against date to see if there are any outliers
fig_raw, ax = plt.subplots()
ax.plot(e_df['allDate'], e_df['eU'])
ax.set_xlabel('Date')
ax.set_ylabel('Energy use (kWh)')

In [ ]:
n_before = len(e_df)
eu_mean = --
eu_std = --
upper = --
lower = --
print('mean', eu_mean, 'std', eu_std)
print('keep eU between', lower, 'and', upper)

n_high = --
n_low = --
e_df = --
n_removed = --
print(n_high, 'rows above,', n_low, 'rows below,', n_removed, 'removed,', len(e_df), 'remain')

Look at the rows that were removed. Most of them have something in common that a real building cannot do. What is it, and what could have happened to the meter?

*Your answer:*

**Question 1, continued — Use the rule**

Use `eu_mean` and `eu_std` from Question 1 (the values before anything was removed).

- `z_11000`: how many standard deviations above the mean is a day with `eU` = 11,000 kWh?
- `outlier_11000`: does the 3 standard deviation rule remove that day? Store `'yes'` or `'no'`.
- `lower_2sd`: the lower threshold if the rule were 2 standard deviations instead of 3.

In [ ]:
z_11000 = --
outlier_11000 = --
lower_2sd = --
print('11,000 kWh is', z_11000, 'standard deviations above the mean. Outlier?', outlier_11000)
print('Lower threshold with 2 standard deviations:', lower_2sd)

**Question 2 — Interaction term and standardizing**

1. Add the column `Temp_x_Med2`: `avgTemp` times the **square** of `MonMed`.
2. Store `eu_mean_clean` and `eu_std_clean`: the mean and standard deviation of `eU` now that the outliers are gone. You need them later to turn standardized values back into kWh.
3. **Standardize** `eU`, `MonMed`, `avgTemp` and `Temp_x_Med2`: subtract the column's mean and divide by its standard deviation.

*Hint: the tutorial's loop normalizes with the minimum and maximum. Here the loop needs the mean and the standard deviation of each column instead.*

In [ ]:
e_df['Temp_x_Med2'] = --

eu_mean_clean = --
eu_std_clean = --

for col in ['eU', 'MonMed', 'avgTemp', 'Temp_x_Med2']:
    e_df[col] = --

e_df[['eU', 'MonMed', 'avgTemp', 'Temp_x_Med2']].describe()

Check your work in the table above without making a plot: what should the mean and the std of a standardized column be? Why are the min and max not 0 and 1?

*Your answer:*

**Question 2, continued — Between kWh and standardized values**

Use `eu_mean_clean` and `eu_std_clean`.

- `z_8000`: the standardized value of a day with `eU` = 8,000 kWh
- `kwh_at_minus_2`: the energy use in kWh of a day whose standardized `eU` is −2

In [ ]:
z_8000 = --
kwh_at_minus_2 = --
print('8,000 kWh standardized:', z_8000)
print('Standardized -2 in kWh:', kwh_at_minus_2)

**Question 3 — Dummy variables**

Create `e_df_dummy` with `pd.get_dummies`: dummy-code `classSession`, `dayType` and `season`, dropping the first level of each. Then read the table:

- `n_columns`: the number of columns of `e_df_dummy`
- `baseline_season`: the season that has no column of its own, as text (for example `'Spring'`)
- `baseline_daytype`: the day type that has no column of its own (`'wk'` or `'wknd'`)
- `row_summer_wknd`: the values of the five dummy columns, in the order `classSession_1`, `dayType_wknd`, `season_Spring`, `season_Summer`, `season_Winter`, for a **weekend day in summer with classes not in session**, as a list of 0s and 1s

In [ ]:
e_df_dummy = pd.get_dummies(--, columns=--, drop_first=--)
print(e_df_dummy.columns)

n_columns = --
baseline_season = --
baseline_daytype = --
row_summer_wknd = --
e_df_dummy.head()

`season` has four values but only three columns. How does the model know that a day is in the baseline season?

*Your answer:*

**Question 4 — Fit the model**

Write the steps yourself; you can name the in-between variables as you like.

1. `y` is the `eU` column of `e_df_dummy`; `x` is everything else
2. Split with **25%** of the rows in the testing set and `random_state=40`
3. Keep the dates of the training rows in `x_train_date`, then drop `allDate` from the training and the testing x (a date is not a predictor)
4. Fit a `LinearRegression` model named `lm` to the training data and predict the training and the testing y as `y_train_predicted` and `y_test_predicted`

Then store `n_train` (the number of training rows) and the R² on the training and the testing data.

In [ ]:
# Separate x and y, split, keep the dates, drop allDate, fit lm, and predict.
y = -- # eU
x = -- # everything else
x_train, x_test, y_train, y_test = train_test_split(--, --, test_size=0.25, random_state=40)
x_train_date = x_train['allDate']
x_train = x_train.drop(columns=['allDate'])
x_test = x_test.drop(columns=['allDate'])
lm = LinearRegression()
lm.fit(x_train, y_train)
y_train_predicted = lm.predict(x_train)
y_test_predicted = lm.predict(x_test)

n_train = --
r2_train = --
r2_test = --
print(n_train, 'training rows')
print('train R2', r2_train)
print('test  R2', r2_test)

**Question 4, continued — What the betas say**

If you did everything correctly, you should have the following values for the betas when you run print(pd.Series(lm.coef_, index=lm.feature_names_in_))

| Feature | Coefficient |
|---|---:|
| `avgTemp` | 0.001798 |
| `MonMed` | 0.687149 |
| `Temp_x_Med2` | 0.035044 |
| `classSession_1` | 0.652321 |
| `dayType_wknd` | -1.042473 |
| `season_Spring` | 0.319108 |
| `season_Summer` | 0.657376 |
| `season_Winter` | 0.259136 |

`lm.coef_` has one beta per column of the training x, in the same order as its columns. Every beta is in standardized units of `eU`, because `eU` was standardized.

- `beta_wknd`: the beta of `dayType_wknd`
- `wknd_kwh`: the same effect in **kWh**: how much more (or less, if negative) energy the model predicts for a weekend day than for a weekday, everything else equal
- `summer_minus_winter`: the predicted standardized `eU` of a summer day minus that of a winter day, everything else equal
- `r2_test_adj`: the adjusted R² on the testing data (formula in Lab 5)

**Worked example: from a beta back to kWh**

Take `season_Spring`, with a beta of 0.319.

1. *What it compares.* A dummy beta is a difference from the baseline category. Fall is the baseline, so a spring day is predicted 0.319 **higher than a fall day**, everything else equal.
2. *Its units.* `eU` was standardized, so 0.319 is in standard deviations of `eU`, not in kWh.
3. *Back to kWh.* One standard deviation of `eU` is `eu_std_clean` kWh, so the difference is `0.319 * eu_std_clean`, about 372 kWh.

Do **not** add `eu_mean_clean` here. In Question 2, continued, you turned one day's *value* back into kWh: z × std + mean. A beta is a *difference* between two days. Both days carry the same mean, so it cancels when you subtract, and only the std is left.

*Hint: one standardized unit of `eU` is one standard deviation of `eU`. Which season do the betas of `season_Summer` and `season_Winter` compare with?*

In [ ]:
print(pd.Series(lm.coef_, index=lm.feature_names_in_))

beta_wknd = lm.coef_[list(x_train.columns).index('dayType_wknd')] # extract the beta of dayType_wknd
wknd_kwh = -- # beta of dayType_wknd * std of eU clean 
summer_minus_winter = -- # beta of season_Summer - beta of season_Winter
r2_test_adj = -- # adjusted R2 of the testing data
print('Weekend beta', beta_wknd, '=', wknd_kwh, 'kWh')
print('Summer minus winter', summer_minus_winter)
print('Test adjusted R2', r2_test_adj)

`avgTemp` and `MonMed` were standardized too. Why does that make their betas easier to compare with each other than the betas of the raw columns would be?

*Your answer:*

**Question 5 — Time-series plot**

Plot the actual and the predicted standardized energy use of the **training** rows against their dates, as points, and show only 1 January to 1 March 2014. Label both axes and add a title.

In [ ]:
fig_ts, ax = plt.subplots()
ax.plot(x_train_date, --, ls='', marker='o', label='Actual')
ax.plot(x_train_date, --, ls='', marker='o', label='Predicted')
fig_ts.autofmt_xdate()
ax.set_xlim(dt.date(2014, 1, 1), dt.date(2014, 3, 1))
ax.set_xlabel('Date')
ax.set_ylabel('Standardized energy use')
ax.set_title('ECJ energy use, training rows')
ax.legend()
fig_ts.savefig(FIGURES_DIR / 'lab06_activity_timeseries.png', dpi=200)

Where in the two months does the model miss the most? What might explain it?

*Your answer:*

**Quiz scratch space**

The Canvas quiz asks about the values above. Use the cell below during the quiz.

**DO NOT OVERWRITE ANY VARIABLES ABOVE THIS CELL. CREATE NEW VARIABLES AS NEEDED.**

In [ ]:
# Use this cell for quiz calculations

## Wrap-up

**Runtime > Restart session and run all**, confirm every cell runs top to bottom, then **File > Download > Download .ipynb** and upload it to Gradescope for feedback. The submission is not graded. Keep this notebook open for the Canvas quiz.